<a href="https://colab.research.google.com/github/saadhana192465019/Digital-Forensics-and-cyber-crime-Investigation--CSA6102/blob/main/Experiment_33_Port_Scan_Detector_from_Packet_Metadata.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================================
# Experiment 5: Port Scan Detector from Packet Metadata
# ==========================================================

from datetime import datetime, timedelta

# Timestamp format
PKT_FMT = "%Y-%m-%d %H:%M:%S"


# ----------------------------------------------------------
# Function to detect port scans
# ----------------------------------------------------------
def detect_port_scan(packets, port_threshold=10, window_seconds=30):
    """
    Detect source IPs scanning multiple destination ports
    within a short time window.
    """

    # Sort packets by time
    packets = sorted(
        packets,
        key=lambda p: datetime.strptime(p["timestamp"], PKT_FMT)
    )

    # Group packets by (source, destination)
    by_pair = {}

    for packet in packets:
        key = (packet["src_ip"], packet["dst_ip"])
        by_pair.setdefault(key, []).append(packet)

    results = {}

    # Check each communication pair
    for key, pkt_list in by_pair.items():

        for i in range(len(pkt_list)):

            start = datetime.strptime(
                pkt_list[i]["timestamp"],
                PKT_FMT
            )

            end = start + timedelta(seconds=window_seconds)

            ports = {
                p["dst_port"]
                for p in pkt_list
                if start <= datetime.strptime(
                    p["timestamp"],
                    PKT_FMT
                ) <= end
            }

            if len(ports) >= port_threshold:

                results[key] = {
                    "distinct_ports": len(ports),
                    "ports": sorted(ports)
                }

                break

    return results


# ==========================================================
# Test Cases
# ==========================================================

def test_experiment5():

    packets = []

    base = datetime(2026, 3, 1, 10, 0, 0)

    # Simulated port scan
    for i, port in enumerate(range(20, 32)):
        packets.append({
            "src_ip": "203.0.113.99",
            "dst_ip": "10.0.0.10",
            "dst_port": port,
            "timestamp": (
                base + timedelta(seconds=2 * i)
            ).strftime(PKT_FMT)
        })

    # Normal HTTPS traffic
    for i in range(5):
        packets.append({
            "src_ip": "10.0.0.20",
            "dst_ip": "10.0.0.30",
            "dst_port": 443,
            "timestamp": (
                base + timedelta(seconds=5 * i)
            ).strftime(PKT_FMT)
        })

    # Detect scans
    results = detect_port_scan(
        packets,
        port_threshold=10,
        window_seconds=30
    )

    # Display Results
    print("Detected Port Scans")
    print("-" * 60)

    if results:
        for (src, dst), info in results.items():
            print(f"Source IP       : {src}")
            print(f"Destination IP  : {dst}")
            print(f"Distinct Ports  : {info['distinct_ports']}")
            print(f"Ports Scanned   : {info['ports']}")
            print()
    else:
        print("No port scan detected.")

    # Assertions
    assert ("203.0.113.99", "10.0.0.10") in results
    assert results[("203.0.113.99", "10.0.0.10")]["distinct_ports"] >= 10
    assert ("10.0.0.20", "10.0.0.30") not in results

    print("All test cases passed.")


# ----------------------------------------------------------
# Run Test
# ----------------------------------------------------------

test_experiment5()

Detected Port Scans
------------------------------------------------------------
Source IP       : 203.0.113.99
Destination IP  : 10.0.0.10
Distinct Ports  : 12
Ports Scanned   : [20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]

All test cases passed.
